# Instalação  e configuração

In [20]:
%pip install ultralytics pyyaml

Note: you may need to restart the kernel to use updated packages.


# Validação de imagens do dataset

Nesta etapa, verificamos:
- imagens sem arquivo `.txt`;
- anotações sem imagem correspondente;
- imagens corrompidas;
- formato YOLO com cinco campos;
- classes 0 e 1;
- coordenadas normalizadas entre 0 e 1;
- quantidade de imagens positivas e negativas.

In [21]:
%pip install pillow

Note: you may need to restart the kernel to use updated packages.


In [22]:
from pathlib import Path
from collections import Counter
from PIL import Image, UnidentifiedImageError


DATASET = Path("/home/emanuele/Documentos/ProjetosFaculdade/urbansense/data/Potholes_dataset")
images = sorted(DATASET.rglob("*.jpg"))
labels = sorted(DATASET.rglob("*.txt"))

missing_labels = []
missing_images = []
invalid_labels = []
corrupt_images = []
class_counts = Counter()
images_with_objects = 0
images_without_objects = 0
invalid_coordinates = 0

for image_path in images:
    label_path = image_path.with_suffix(".txt")

    if not label_path.exists():
        missing_labels.append(str(image_path))
        continue

    try:
        with Image.open(image_path) as image:
            image.verify()
    except (UnidentifiedImageError, OSError):
        corrupt_images.append(str(image_path))

    lines = [
        line.strip()
        for line in label_path.read_text().splitlines()
        if line.strip()
    ]

    if not lines:
        images_without_objects += 1
    else:
        images_with_objects += 1

    for line_number, line in enumerate(lines, start=1):
        fields = line.split()

        if len(fields) != 5:
            invalid_labels.append(
                f"{label_path}:{line_number} - campos: {len(fields)}"
            )
            continue

        try:
            class_id = int(fields[0])
            x, y, width, height = map(float, fields[1:])
        except ValueError:
            invalid_labels.append(
                f"{label_path}:{line_number} - valor inválido"
            )
            continue

        if class_id not in (0, 1):
            invalid_labels.append(
                f"{label_path}:{line_number} - classe inválida: {class_id}"
            )

        if not all(0 <= value <= 1 for value in (x, y, width, height)):
            invalid_coordinates += 1
            invalid_labels.append(
                f"{label_path}:{line_number} - coordenada fora de [0, 1]"
            )

        if width <= 0 or height <= 0:
            invalid_labels.append(
                f"{label_path}:{line_number} - largura/altura inválida"
            )

        class_counts[class_id] += 1

label_images = {path.with_suffix(".jpg") for path in labels}
missing_images = [
    str(path)
    for path in label_images
    if not path.exists()
]

print("Imagens:", len(images))
print("Anotações:", len(labels))
print("Imagens com objetos:", images_with_objects)
print("Imagens negativas:", images_without_objects)
print("Caixas por classe:", dict(class_counts))
print("Imagens sem anotação:", len(missing_labels))
print("Anotações sem imagem:", len(missing_images))
print("Imagens corrompidas:", len(corrupt_images))
print("Anotações inválidas:", len(invalid_labels))
print("Coordenadas fora do intervalo:", invalid_coordinates)

assert not missing_labels, "Existem imagens sem arquivo de anotação"
assert not missing_images, "Existem anotações sem imagem correspondente"
assert not corrupt_images, "Existem imagens corrompidas"
assert not invalid_labels, "Existem anotações inválidas"

Imagens: 2099
Anotações: 2099
Imagens com objetos: 1510
Imagens negativas: 589
Caixas por classe: {0: 3166, 1: 442}
Imagens sem anotação: 0
Anotações sem imagem: 0
Imagens corrompidas: 0
Anotações inválidas: 0
Coordenadas fora do intervalo: 0


# Divisão treino/validação/teste

70/15/15.

In [23]:
%pip install scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [24]:
from sklearn.model_selection import train_test_split
from pathlib import Path
import shutil

DATASET_SAIDA = DATASET.parent / "Potholes_dataset_dividido"
SEMENTE = 42

images = sorted(DATASET.rglob("*.jpg"))
print("Imagens para dividir:", len(images))

registros = []

for imagem in images:
    anotacao = imagem.with_suffix(".txt")
    classes = sorted({
        linha.split()[0]
        for linha in anotacao.read_text().splitlines()
        if linha.strip()
    })
    assinatura = "|".join(classes) if classes else "sem_objeto"
    registros.append((imagem, assinatura))

print("Registros para dividir:", len(registros))

assinaturas = [assinatura for _, assinatura in registros]

treino, temporario = train_test_split(
    registros,
    test_size=0.30,
    random_state=SEMENTE,
    stratify=assinaturas
)

assinaturas_temporarias = [assinatura for _, assinatura in temporario]

validacao, teste = train_test_split(
    temporario,
    test_size=0.50,
    random_state=SEMENTE,
    stratify=assinaturas_temporarias
)

particoes = {
    "train": treino,
    "val": validacao,
    "test": teste,
}

for nome, registros_particao in particoes.items():
    for imagem, _ in registros_particao:
        caminho_relativo = imagem.relative_to(DATASET)
        anotacao = imagem.with_suffix(".txt")

        destino_imagem = DATASET_SAIDA / "images" / nome / caminho_relativo
        destino_anotacao = DATASET_SAIDA / "labels" / nome / caminho_relativo.with_suffix(".txt")

        destino_imagem.parent.mkdir(parents=True, exist_ok=True)
        destino_anotacao.parent.mkdir(parents=True, exist_ok=True)

        shutil.copy2(imagem, destino_imagem)
        shutil.copy2(anotacao, destino_anotacao)

for nome, registros_particao in particoes.items():
    print(f"{nome}: {len(registros_particao)} imagens")

Imagens para dividir: 2099
Registros para dividir: 2099
train: 1469 imagens
val: 315 imagens
test: 315 imagens


In [25]:
from collections import Counter

# Confirma cobertura total e distribuição de classes em cada partição.
total_dividido = sum(len(registros) for registros in particoes.values())
print("Total dividido:", total_dividido)
print("Data original:", len(images))
print("Cobertura completa:", total_dividido == len(images))

for nome, registros_particao in particoes.items():
    classes = Counter()
    for imagem, _ in registros_particao:
        anotacao = imagem.with_suffix(".txt")
        classes.update(
            linha.split()[0]
            for linha in anotacao.read_text().splitlines()
            if linha.strip()
        )
    print(nome, dict(classes))

Total dividido: 2099
Data original: 2099
Cobertura completa: True
train {'0': 2240, '1': 312}
val {'1': 66, '0': 459}
test {'1': 64, '0': 467}


In [26]:
import csv

manifesto = DATASET_SAIDA / "manifest.csv"

with manifesto.open("w", newline="", encoding="utf-8") as arquivo:
    escritor = csv.writer(arquivo)
    escritor.writerow(["imagem", "split", "anotacao"])

    for nome, registros_particao in particoes.items():
        for imagem, _ in registros_particao:
            anotacao = imagem.with_suffix(".txt")
            escritor.writerow([
                str(imagem.relative_to(DATASET)),
                nome,
                str(anotacao.relative_to(DATASET)),
            ])

print("Manifesto salvo em:", manifesto)

Manifesto salvo em: /home/emanuele/Documentos/ProjetosFaculdade/urbansense/data/Potholes_dataset_dividido/manifest.csv


In [27]:
import yaml

configuracao = {
    "path": str(DATASET_SAIDA.resolve()),
    "train": "images/train",
    "val": "images/val",
    "test": "images/test",
    "names": {
        0: "pothole",
        1: "manhole_cover",
    },
}

yaml_path = DATASET_SAIDA / "data.yaml"
yaml_path.write_text(
    yaml.safe_dump(configuracao, sort_keys=False),
    encoding="utf-8",
)

print(yaml_path.read_text())

path: /home/emanuele/Documentos/ProjetosFaculdade/urbansense/data/Potholes_dataset_dividido
train: images/train
val: images/val
test: images/test
names:
  0: pothole
  1: manhole_cover



In [28]:
for nome in ("train", "val", "test"):
    imagens_split = sorted(
        (DATASET_SAIDA / "images" / nome).rglob("*.jpg")
    )
    anotacoes_split = sorted(
        (DATASET_SAIDA / "labels" / nome).rglob("*.txt")
    )
    print(nome, len(imagens_split), "imagens,", len(anotacoes_split), "anotações")

train 1469 imagens, 1469 anotações
val 315 imagens, 315 anotações
test 315 imagens, 315 anotações


# Baseline de treino

%pip install ultralytics

In [29]:
from ultralytics import YOLO

baseline = YOLO("yolo11n.pt")

historico_baseline = baseline.train(
    data=str(DATASET_SAIDA / "data.yaml"),
    epochs=50,
    imgsz=640,
    batch=8,
    device="cpu",
    seed=42,
    project="runs",
    name="baseline-cpu",
    exist_ok=True,
)

baseline = YOLO("runs/baseline-cpu/weights/best.pt")

metricas_baseline = baseline.val(
    data=str(DATASET_SAIDA / "data.yaml"),
    split="test",
    imgsz=640,
    batch=8,
    device="cpu",
)

print("mAP50:", metricas_baseline.box.map50)
print("mAP50-95:", metricas_baseline.box.map)
print("Precisão:", metricas_baseline.box.mp)
print("Recall:", metricas_baseline.box.mr)



Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/home/emanuele/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
Ultralytics 8.4.216 🚀 Python-3.10.12 torch-2.14.1+cu130 CPU (Intel Core 5 120U)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/emanuele/Documentos/ProjetosFaculdade/urbansense/data/Potholes_dataset_dividido/data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, ex

KeyboardInterrupt: 